**RAG Based Wikipedia Knowledge Base Search Article**
---



## Install required libraries:

wikipedia  -> Collect Wikipedia articles
  
pandas     -> Store article metadata in tabular format

tqdm       -> Display progress bars during data collection

In [1]:
!pip3 install wikipedia pandas tqdm

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=5657751b430ba60bd54e1f0fa47aabdba5217e72b8b2c2add7cab149e5a88337
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia


In [2]:
import os
import time
import requests
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# =====================================================
# Configuration - "This section sets up the project environment and defines configuration parameters that control article collection."
# =====================================================

# Folder where downloaded Wikipedia articles will be stored
OUTPUT_DIR = Path("AI_Knowledge_Base")
OUTPUT_DIR.mkdir(exist_ok=True)

# User-Agent helps identify the application making requests
HEADERS = {
    "User-Agent": "AIML-RAG-Dataset-Builder/1.0"
}

# Number of Wikipedia search results to retrieve
SEARCH_LIMIT = 10

# Retry failed requests up to 3 times
MAX_RETRIES = 3

# Delay between requests to avoid overwhelming the API
SLEEP_TIME = 1

# =====================================================
# Seed Keywords
# =====================================================

KEYWORDS = [

    "Artificial Intelligence",
    "Machine Learning",
    "Deep Learning",
    "Large Language Model",
    "Natural Language Processing",
    "Data Science",
    "Computer Vision",
    "Generative AI",
    "Transformer",
    "Neural Network",
    "Prompt Engineering",
    "Retrieval Augmented Generation",
    "Semantic Search",
    "Vector Database",
    "Feature Engineering"

]

# =====================================================
# Search Wikipedia
# =====================================================

def search_articles(keyword, limit=10):

    url = "https://en.wikipedia.org/w/api.php"

    params = {
        "action": "query",
        "list": "search",
        "srsearch": keyword,
        "srlimit": limit,
        "format": "json"
    }

    response = requests.get(
        url,
        params=params,
        headers=HEADERS,
        timeout=30
    )

    response.raise_for_status()

    data = response.json()

    titles = []

    for article in data["query"]["search"]:
        titles.append(article["title"])

    return titles

# =====================================================
# Download Wikipedia Article
# =====================================================

def download_article(title):

    url = "https://en.wikipedia.org/w/api.php"

    params = {

        "action": "query",
        "format": "json",
        "titles": title,
        "redirects": 1,
        "prop": "extracts|info",
        "inprop": "url",
        "explaintext": 1

    }

    response = requests.get(
        url,
        params=params,
        headers=HEADERS,
        timeout=30
    )

    response.raise_for_status()

    data = response.json()

    page = list(data["query"]["pages"].values())[0]

    if "missing" in page:
        return None

    return {

        "title": page["title"],
        "text": page.get("extract", ""),
        "url": page.get("fullurl", "")

    }

# =====================================================
# Safe filename
# =====================================================

def clean_filename(name):

    invalid = '<>:"/\\|?*'

    for ch in invalid:
        name = name.replace(ch, "_")

    return name

# =====================================================
# Main Download Loop
# =====================================================

metadata = []
errors = []

visited = set()

print("="*60)
print("Searching Wikipedia")
print("="*60)

candidate_titles = []

for keyword in KEYWORDS:

    print(f"\nSearching : {keyword}")

    try:

        results = search_articles(keyword, SEARCH_LIMIT)

        candidate_titles.extend(results)

        print(f"Found {len(results)} articles")

    except Exception as e:

        print(e)

candidate_titles = sorted(set(candidate_titles))

print("\n")
print("="*60)
print(f"Total Unique Articles Found : {len(candidate_titles)}")
print("="*60)

# =====================================================
# Download
# =====================================================

for title in tqdm(candidate_titles):

    if title in visited:
        continue

    visited.add(title)

    success = False

    for retry in range(MAX_RETRIES):

        try:

            page = download_article(title)

            if page is None:
                raise Exception("Page not found")

            filename = clean_filename(page["title"]) + ".txt"

            filepath = OUTPUT_DIR / filename

            if filepath.exists():

                success = True

                break

            with open(
                filepath,
                "w",
                encoding="utf-8"
            ) as f:

                f.write(page["text"])

            metadata.append({

                "Title": page["title"],
                "URL": page["url"],
                "Words": len(page["text"].split()),
                "Characters": len(page["text"]),
                "File": filename

            })

            success = True

            break

        except Exception as e:

            if retry == MAX_RETRIES - 1:

                errors.append({

                    "Title": title,
                    "Error": str(e)

                })

            time.sleep(SLEEP_TIME)

# =====================================================
# Save CSV
# =====================================================

metadata_df = pd.DataFrame(metadata)

metadata_df.to_csv(
    OUTPUT_DIR / "metadata.csv",
    index=False
)

errors_df = pd.DataFrame(errors)

errors_df.to_csv(
    OUTPUT_DIR / "error_log.csv",
    index=False
)

print("\n")
print("="*60)
print("Download Completed")
print("="*60)
print(f"Articles Downloaded : {len(metadata_df)}")
print(f"Failed Downloads    : {len(errors_df)}")
print(f"Folder              : {OUTPUT_DIR}")
print("="*60)

Searching Wikipedia

Searching : Artificial Intelligence
Found 10 articles

Searching : Machine Learning
Found 10 articles

Searching : Deep Learning
Found 10 articles

Searching : Large Language Model
Found 10 articles

Searching : Natural Language Processing
Found 10 articles

Searching : Data Science
Found 10 articles

Searching : Computer Vision
Found 10 articles

Searching : Generative AI
Found 10 articles

Searching : Transformer
Found 10 articles

Searching : Neural Network
Found 10 articles

Searching : Prompt Engineering
429 Client Error: Too Many Requests for url: https://en.wikipedia.org/w/api.php?action=query&list=search&srsearch=Prompt+Engineering&srlimit=10&format=json

Searching : Retrieval Augmented Generation
429 Client Error: Too Many Requests for url: https://en.wikipedia.org/w/api.php?action=query&list=search&srsearch=Retrieval+Augmented+Generation&srlimit=10&format=json

Searching : Semantic Search
429 Client Error: Too Many Requests for url: https://en.wikipedia.o

100%|██████████| 88/88 [03:05<00:00,  2.11s/it]




Download Completed
Articles Downloaded : 36
Failed Downloads    : 52
Folder              : AI_Knowledge_Base


**Install Libraries**

In [1]:

!pip install langchain
!pip install langchain-community
!pip install langchain-huggingface
!pip install sentence-transformers
!pip install faiss-cpu
!pip install transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 94.8 MB/s eta 0:00:00


**Load Wikipedia Text Files**

This step loads the raw Wikipedia articles and converts them into LangChain Document objects for further processing.

In [2]:
# Import LangChain Document object
# Each Wikipedia article will be stored as a Document

import os
from langchain_core.documents import Document

documents = []

folder_path = "AI_Knowledge_Base"

# Read all text files from the knowledge base folder
for file in os.listdir(folder_path):

    if file.endswith(".txt"):

        with open(
            os.path.join(folder_path, file),
            "r",
            encoding="utf-8"
        ) as f:

            text = f.read()

            # Convert text into LangChain Document format
            # Metadata stores the source file name
            documents.append(
                Document(
                    page_content=text,
                    metadata={"source": file}
                )
            )

print("Documents Loaded:", len(documents))

Documents Loaded: 36


**Split Documents into Chunks**

Chunking improves retrieval quality because the vector database searches smaller pieces of text instead of entire documents.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Split long documents into smaller chunks
# LLMs perform better when retrieving smaller relevant passages

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # Maximum characters per chunk
    chunk_overlap=50  # Shared text between chunks
)

chunks = text_splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

Total Chunks: 2378


**Create Embeddings**

Embeddings transform text into vector representations so semantic similarity can be calculated.

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

# Load sentence embedding model
# Converts text into numerical vectors
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

**Create FAISS Vector Store**

FAISS stores vector embeddings and enables efficient similarity search during retrieval.

In [5]:
from langchain_community.vectorstores import FAISS

# Create vector database from document chunks
# Each chunk is converted into an embedding vector

vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

# Save the vector database for future use
vectorstore.save_local("wiki_faiss_db")

print("FAISS Database Created")


/tmp/ipykernel_4926/870739733.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


FAISS Database Created


**Load Existing Vector Store**

This step restores the vector database from disk, reducing startup time.

In [6]:
# Load previously saved FAISS index
# Prevents rebuilding embeddings every time
vectorstore = FAISS.load_local(
    "wiki_faiss_db",
    embeddings,
    allow_dangerous_deserialization=True
)


**Create Retriever**

The retriever returns the top 3 most relevant chunks for a given query

In [7]:
# Create retriever object
# Retriever searches the vector database for relevant chunks
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)


**Test Retrieval**

This step validates whether the vector database is returning relevant information."

In [8]:
# Example query to verify retrieval quality
query = "What is Machine Learning?"

# Retrieve top matching chunks
docs = retriever.invoke(query)

# Display retrieved content
for i, doc in enumerate(docs):
    print(f"\nDocument {i+1}")
    print(doc.page_content[:500])



Document 1
=== Learning ===
Machine learning is the study of programs that can improve their performance on a given task automatically. It has been a part of AI from the beginning.

There are several kinds of machine learning:

Document 2
=== Learning paradigms ===
Machine learning has involved a variety of approaches to training models, including supervised learning, unsupervised learning, reinforcement learning, and self-supervised learning.

Document 3
Machine learning is used in diverse types of reverse engineering. For example, machine learning has been used to reverse engineer a composite material part, enabling unauthorized production of high quality parts, and for quickly understanding the behavior of malware. It can be used to reverse engineer artificial intelligence models. It can also design components by engaging in a type of reverse engineering of not-yet existent virtual components such as inverse molecular design for particular


**Load LLM**

The LLM is responsible for generating natural-language answers based on retrieved knowledge

In [9]:
from transformers import pipeline

# Load instruction-tuned TinyLlama model
# This model generates final answers using retrieved context
llm = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    device_map="auto"
)


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

**Build Simple RAG Function**

This function demonstrates the complete RAG workflow: Retrieve → Augment Prompt → Generate Answer

In [10]:
# Function implementing the Retrieval-Augmented Generation pipeline

def ask_rag(question):

    # Step 1: Retrieve top 3 relevant chunks
    docs = vectorstore.similarity_search(
        question,
        k=3
    )

    # Step 2: Combine retrieved chunks into context
    context = "\n".join(
        [doc.page_content for doc in docs]
    )

    # Step 3: Create prompt containing context and question
    prompt = f"""
    Context:
    {context}

    Question:
    {question}

    Answer:
    """


    # Step 4: Generate answer using the LLM
    response = llm(
        prompt,
        max_new_tokens=200,
        temperature=0.7
    )

    return response[0]["generated_text"]

**Ask Questions**

This is the final execution step where a user query is processed through retrieval and answer generation.

In [11]:
# Example user question
question = "What is Natural Language Processing?"


# Generate answer using RAG pipeline
answer = ask_rag(question)


# Display final response
print(answer)


[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



    Context:
    Natural language processing (NLP) is the processing of natural language information by a computer. NLP is a subfield of computer science and is closely associated with artificial intelligence. NLP is also related to information retrieval, knowledge representation, computational linguistics, and linguistics more broadly.
Major processing tasks in an NLP system include: speech recognition, text classification, natural language understanding, and natural language generation.


== History ==
=== Natural language processing ===
Natural language processing (NLP) allows programs to read, write, and communicate in human languages. Specific problems include speech recognition, speech synthesis, machine translation, information extraction, information retrieval, and question answering.
The umbrella term "natural language understanding" can be applied to a diverse set of computer applications, ranging from small, relatively simple tasks such as short commands issued to robots, t